## Breakpoint Analysis (Whole Corpus)

### Aim
Our earlier correlations and descriptive trends were computed on the **full 10-K text corpus**. To avoid testing a different object (e.g., Risk Factors only), the breakpoint hypothesis must be evaluated on a **firm–year corpus-level ESG signal** aggregated across all available sections.

### Hypotheses
- **H1 (Paris/2015 break):** ESG disclosure intensity shifts upward starting **2015/2016**.
- **H2 (2020–2021 break):** the dominant regime shift occurs around **2020–2021**.

### Corpus-Level Panel Construction
Build a `firm_year_corpus` dataset (one row per firm-year) containing:
- `E_hits`, `S_hits`, `G_hits` (lexicon term match counts across the entire corpus)
- `tokens` (total token count across all sections)
- `E_rate`, `S_rate`, `G_rate` = hits per 1,000 tokens (length-normalised intensity)
- `n_sections` (how many distinct sections are present per firm-year, for coverage / robustness)
- identifiers: `ticker`, `cik`, `year`, `gics_sector`

This corpus approach reduces section-specific bias and aligns the breakpoint test with the scope used in the correlation stage.

### Core Tests
- Breakpoint regressions comparing candidate breaks (2015/16 vs 2020/21) using the same dependent variables (`E_rate`, `S_rate`, `G_rate`).
- Piecewise trend models to compare break magnitudes and model fit (AIC/BIC).

### Robustness
- Placebo breaks (2017–2019) to show the effect is not generic time drift.
- Sector interactions to test heterogeneity across GICS sectors.
- Term ablation (remove most common terms and re-run) to reduce boilerplate sensitivity.

### Deliverables
- Plot: mean `E_rate` over time with break lines at 2015 and 2021 (with uncertainty bands).
- Table: break coefficients + AIC/BIC comparisons for each candidate breakpoint.
- Placebo plot: estimated “break effect” by candidate year.


In [13]:
# Imports
from pathlib import Path
import polars as pl
import yaml

# --------------------
# Output helpers (tables -> SQLite)
# --------------------
OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

DB_PATH = OUT_DIR / "esg_panel.sqlite"
DB_URI = f"sqlite:///{DB_PATH}"  # SQLAlchemy-style URI Polars expects

def save_table_sqlite(df: pl.DataFrame, table_name: str) -> Path:
    # Overwrite table each run
    df.write_database(
        table_name=table_name,
        connection=DB_URI,
        if_table_exists="replace",
    )
    print(f"Saved table → {DB_PATH} (table: {table_name})")
    return DB_PATH

# Optional: create a couple of useful indexes (SQLite setup)
def sqlite_setup_indexes() -> None:
    import sqlite3
    con = sqlite3.connect(DB_PATH)
    cur = con.cursor()

    # Fast lookups / joins
    cur.execute("CREATE INDEX IF NOT EXISTS idx_fyc_ticker_year ON firm_year_corpus(ticker, year);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_fyc_cik_year ON firm_year_corpus(cik, year);")
    cur.execute("CREATE INDEX IF NOT EXISTS idx_fyc_sector_year ON firm_year_corpus(gics_sector, year);")

    con.commit()
    con.close()
    print("SQLite setup → indexes created/verified")

# --------------------
# Config
# --------------------
PARQUET_FILE = "spy_10k_2015_present.parquet"
LEXICON_FILE = "ESG_Lexicon.yml"

# --------------------
# Load data
# --------------------
df = pl.read_parquet(PARQUET_FILE)

# Basic hygiene
df = df.with_columns(
    pl.col("cik").cast(pl.Utf8),              # Stable key across ticker changes
    pl.col("ticker").cast(pl.Utf8),
    pl.col("company_name").cast(pl.Utf8),
    pl.col("section").cast(pl.Utf8),
    pl.col("text").cast(pl.Utf8),
    pl.col("gics_sector").cast(pl.Utf8),
    pl.col("filing_date").cast(pl.Date),      # Needed for year index
)

# Required columns check
required = {"ticker", "cik", "filing_date", "gics_sector", "section", "text"}
missing = required - set(df.columns)
assert not missing, f"Missing columns: {missing}"

# Year key
df = df.with_columns(
    pl.col("filing_date").dt.year().alias("year")  # Filing year convention
)

# Token count (count runs of non-whitespace)
df = df.with_columns(
    pl.col("text")
      .fill_null("")
      .str.count_matches(r"\S+")
      .alias("tokens")
)

# --------------------
# Load lexicon
# --------------------
with open(LEXICON_FILE, "r", encoding="utf-8") as f:
    lex = yaml.safe_load(f)

E_terms = list(lex.get("environmental") or [])
S_terms = list(lex.get("social") or [])
G_terms = list(lex.get("governance") or [])
assert E_terms and S_terms and G_terms, "Lexicon pillars are empty or missing keys"

def pillar_hits_expr(terms: list[str], text_col: str = "text") -> pl.Expr:
    # Sum regex match counts across all patterns in a pillar
    return pl.sum_horizontal([pl.col(text_col).str.count_matches(t) for t in terms])

# Add section-level hit counts
df = df.with_columns(
    pillar_hits_expr(E_terms).alias("E_hits"),
    pillar_hits_expr(S_terms).alias("S_hits"),
    pillar_hits_expr(G_terms).alias("G_hits"),
)

# --------------------
# Aggregate to firm-year corpus (all sections)
# --------------------
firm_year_corpus = (
    df.group_by(["ticker", "cik", "year", "gics_sector"])
      .agg(
          pl.sum("tokens").alias("tokens"),
          pl.sum("E_hits").alias("E_hits"),
          pl.sum("S_hits").alias("S_hits"),
          pl.sum("G_hits").alias("G_hits"),
          pl.n_unique("section").alias("n_sections"),
      )
      .filter(pl.col("tokens") > 0)  # Avoid divide-by-zero
      .with_columns(
          (pl.col("E_hits") * 1000 / pl.col("tokens")).alias("E_rate"),
          (pl.col("S_hits") * 1000 / pl.col("tokens")).alias("S_rate"),
          (pl.col("G_hits") * 1000 / pl.col("tokens")).alias("G_rate"),
      )
)

# --------------------
# Save outputs (workspace-first: SQLite DB)
# --------------------
save_table_sqlite(firm_year_corpus, "firm_year_corpus")
sqlite_setup_indexes()

# Lightweight console preview (no notebook rendering dependency)
print(
    firm_year_corpus.select([
        "ticker","cik","year","gics_sector","tokens",
        "E_hits","S_hits","G_hits","n_sections","E_rate","S_rate","G_rate"
    ]).head(10)
)


Saved table → outputs\esg_panel.sqlite (table: firm_year_corpus)
SQLite setup → indexes created/verified
shape: (10, 12)
┌────────┬────────────┬──────┬──────────────────┬───┬────────────┬───────────┬──────────┬──────────┐
│ ticker ┆ cik        ┆ year ┆ gics_sector      ┆ … ┆ n_sections ┆ E_rate    ┆ S_rate   ┆ G_rate   │
│ ---    ┆ ---        ┆ ---  ┆ ---              ┆   ┆ ---        ┆ ---       ┆ ---      ┆ ---      │
│ str    ┆ str        ┆ i32  ┆ str              ┆   ┆ u32        ┆ f64       ┆ f64      ┆ f64      │
╞════════╪════════════╪══════╪══════════════════╪═══╪════════════╪═══════════╪══════════╪══════════╡
│ WM     ┆ 0000823768 ┆ 2020 ┆ Industrials      ┆ … ┆ 2          ┆ 17.463574 ┆ 3.471674 ┆ 2.998264 │
│ DIS    ┆ 0001744489 ┆ 2023 ┆ Communication    ┆ … ┆ 2          ┆ 0.972868  ┆ 6.972219 ┆ 1.51335  │
│        ┆            ┆      ┆ Services         ┆   ┆            ┆           ┆          ┆          │
│ TFC    ┆ 0000092230 ┆ 2023 ┆ Financials       ┆ … ┆ 2          ┆ 1.34

In [12]:
uv pip install sqlalchemy

Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.7 environment at: c:\Users\ddddd\AppData\Local\Programs\Python\Python313
Resolved 3 packages in 1.13s
Prepared 2 packages in 1.25s
Installed 3 packages in 194ms
 + greenlet==3.3.1
 + sqlalchemy==2.0.46
 + typing-extensions==4.15.0


In [15]:
from pathlib import Path
import pandas as pd
import polars as pl
from sqlalchemy import create_engine, text
import sqlite3

OUT_DIR = Path("outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Versioned DBs
DB_IN  = OUT_DIR / "esg_panel_01.sqlite"   # input (existing)
DB_OUT = OUT_DIR / "esg_panel_02.sqlite"   # output (new, do not overwrite)

DB_IN_URL  = f"sqlite:///{DB_IN}"
DB_OUT_URL = f"sqlite:///{DB_OUT}"

TABLE_IN  = "firm_year_corpus"
TABLE_OUT = "panel_firm_year_corpus"

# --------------------
# Read from input DB (SQLAlchemy -> pandas -> polars)
# --------------------
engine_in = create_engine(DB_IN_URL)
with engine_in.connect() as conn:
    df_pd = pd.read_sql(text(f"SELECT * FROM {TABLE_IN}"), conn)

firm_year_corpus = pl.from_pandas(df_pd)

# --------------------
# Add panel features
# --------------------
min_year = firm_year_corpus.select(pl.col("year").min()).item()

firm_year_corpus = firm_year_corpus.with_columns(
    (pl.col("year") >= 2015).cast(pl.Int8).alias("post_2015"),
    (pl.col("year") >= 2020).cast(pl.Int8).alias("post_2020"),
    (pl.col("year") >= 2021).cast(pl.Int8).alias("post_2021"),
    (pl.col("year") - pl.lit(min_year)).cast(pl.Int32).alias("t"),
)

# --------------------
# Write to new output DB (no overwrite risk)
# --------------------
firm_year_corpus.write_database(
    table_name=TABLE_OUT,
    connection=DB_OUT_URL,
    if_table_exists="fail",  # fail fast if you accidentally reuse a DB/table
)

print(f"Wrote new DB → {DB_OUT} (table: {TABLE_OUT})")

# --------------------
# Setup indexes in the new DB
# --------------------
con = sqlite3.connect(DB_OUT)
cur = con.cursor()
cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{TABLE_OUT}_ticker_year ON {TABLE_OUT}(ticker, year);")
cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{TABLE_OUT}_cik_year ON {TABLE_OUT}(cik, year);")
cur.execute(f"CREATE INDEX IF NOT EXISTS idx_{TABLE_OUT}_sector_year ON {TABLE_OUT}(gics_sector, year);")
con.commit()
con.close()
print("SQLite setup → indexes created/verified")



DatabaseError: Execution failed on sql 'SELECT * FROM firm_year_corpus': (sqlite3.OperationalError) no such table: firm_year_corpus
[SQL: SELECT * FROM firm_year_corpus]
(Background on this error at: https://sqlalche.me/e/20/e3q8)

In [ ]:
uv pip install python-dotenv pymysql


Note: you may need to restart the kernel to use updated packages.


Using Python 3.13.7 environment at: c:\Users\ddddd\AppData\Local\Programs\Python\Python313
Resolved 1 package in 751ms
Prepared 1 package in 934ms
Installed 1 package in 31ms
 + fastexcel==0.19.0
